# Fun-ASR-Nano 微调与测试 (衡东方言)

微调 Fun-ASR-Nano (800M) 模型，支持湘语（衡东方言）。
需要从 Fun-ASR 仓库运行训练。

## 1. 环境检查

In [ ]:
import torch
import subprocess
import json
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

try:
    import funasr
    print(f"FunASR: {funasr.__version__}")
except ImportError:
    print("FunASR 未安装")

try:
    import torchaudio
    print(f"torchaudio: {torchaudio.__version__}")
except ImportError:
    print("torchaudio 未安装")

## 2. 克隆 Fun-ASR 仓库

In [ ]:
FUNASR_REPO = "/mnt/Fun-ASR"

if not os.path.exists(FUNASR_REPO):
    print("克隆 Fun-ASR 仓库...")
    result = subprocess.run(
        ["git", "clone", "https://github.com/FunAudioLLM/Fun-ASR.git", FUNASR_REPO],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("克隆成功!")
    else:
        print(f"克隆失败: {result.stderr}")
else:
    print(f"Fun-ASR 仓库已存在: {FUNASR_REPO}")

## 3. 数据检查

In [ ]:
TRAIN_JSONL = "/mnt/data/prepared_data/funasr_nano/train.jsonl"
VAL_JSONL = "/mnt/data/prepared_data/funasr_nano/val.jsonl"

for path, label in [(TRAIN_JSONL, '训练集'), (VAL_JSONL, '验证集')]:
    if os.path.exists(path):
        count = sum(1 for _ in open(path))
        print(f"  {label}: {count} 条")
    else:
        print(f"  {label}: 不存在! {path}")

if os.path.exists(TRAIN_JSONL):
    with open(TRAIN_JSONL) as f:
        sample = json.loads(f.readline())
    print(f"\n数据样例:")
    print(json.dumps(sample, ensure_ascii=False, indent=2))

## 4. 训练配置

In [ ]:
OUTPUT_DIR = "/mnt/output/funasr_nano_v3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FUNASR_REPO_DEFAULT = "/mnt/Fun-ASR"
if 'FUNASR_REPO' not in dir():
    FUNASR_REPO = FUNASR_REPO_DEFAULT

import funasr.bin.train_ds
train_ds_path = funasr.bin.train_ds.__file__
print(f"train_ds.py: {train_ds_path}")

DEEPSPEED_CONF = f"{FUNASR_REPO}/deepspeed_conf/ds_stage1.json"
print(f"DeepSpeed配置: {DEEPSPEED_CONF}")

cmd = [
    "torchrun", "--nproc_per_node=1", "--master_port=29501",
    train_ds_path,
    "++model=FunAudioLLM/Fun-ASR-Nano-2512",
    "++trust_remote_code=true",
    f"++train_data_set_list={TRAIN_JSONL}",
    f"++valid_data_set_list={VAL_JSONL}",
    "++dataset_conf.data_split_num=1",
    "++dataset_conf.batch_sampler=BatchSampler",
    "++dataset_conf.batch_size=4000",
    "++dataset_conf.sort_size=1024",
    "++dataset_conf.batch_type=token",
    "++dataset_conf.num_workers=8",
    "++train_conf.max_epoch=10",
    "++train_conf.log_interval=50",
    "++train_conf.resume=true",
    "++train_conf.validate_interval=1000",
    "++train_conf.save_checkpoint_interval=1000",
    "++train_conf.keep_nbest_models=5",
    "++train_conf.avg_nbest_model=3",
    "++train_conf.use_deepspeed=false",
    f"++train_conf.deepspeed_config={DEEPSPEED_CONF}",
    "++train_conf.use_bf16=true",
    "++train_conf.grad_clip=5",
    "++train_conf.effective_save_name_excludes=[]",
    "++optim=adamw",
    "++optim_conf.lr=5e-5",
    "++optim_conf.weight_decay=0.01",
    "++audio_encoder_conf.freeze=true",
    "++audio_adaptor_conf.freeze=false",
    "++llm_conf.freeze=false",
    f"++output_dir={OUTPUT_DIR}",
]

env = os.environ.copy()
env["PYTHONPATH"] = f"{FUNASR_REPO}:{env.get('PYTHONPATH', '')}"

print(f"\n模型: Fun-ASR-Nano-2512 (800M)")
print(f"输出: {OUTPUT_DIR}")
print(f"冻结: audio_encoder=True, llm=False (全量微调)")
print(f"保存: 全部权重")
print()
print("开始训练...")
print("=" * 60)

In [ ]:
import sys

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=FUNASR_REPO, env=env,
)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()

returncode = process.wait()
print(f"\n{'=' * 60}")
if returncode == 0:
    print(f"训练完成! 模型保存在: {OUTPUT_DIR}")
else:
    print(f"训练失败! 返回码: {returncode}")

## 5. 加载模型并测试

In [ ]:
import sys
import re
sys.path.insert(0, FUNASR_REPO)

from funasr import AutoModel

model = AutoModel(
    model="FunAudioLLM/Fun-ASR-Nano-2512",
    trust_remote_code=True,
    hub="ms",
    init_param=f"{OUTPUT_DIR}/model.pt.best",
)
print("Fun-ASR-Nano 加载成功!")

# 单条测试
samples = []
with open(VAL_JSONL) as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        samples.append(json.loads(line))

print(f"\n单条测试 ({len(samples)} 条):")
for s in samples[:5]:
    user_content = s['messages'][1]['content']
    match = re.search(r'!(.*?)<\|endofspeech\|>', user_content)
    audio = match.group(1) if match else None
    expected = s['messages'][2]['content']
    if not audio or not os.path.exists(audio):
        continue
    result = model.generate(input=audio, language="中文", itn=True)
    pred = result[0]['text'] if result else ""
    flag = "V" if expected in pred else "X"
    print(f"  [{flag}] {expected} → {pred}")